# ⚙️ Feature Engineering Pipeline as Versioned Artifact

**Project 3 of 10 — MLOps Portfolio Series**  
**Author:** Jumma Mohammad Teli | Birmingham, UK  
**Dataset:** UCI Bank Marketing (41,188 rows)  
**Goal:** Package feature engineering as a versioned sklearn artifact, decoupled from the model

---

## Notebook Structure
1. Setup & Imports
2. Raw Data Exploration
3. Transformer Deep Dives
4. Full Pipeline Execution
5. Feature Impact Analysis
6. MLflow Artifact Logging
7. Decoupling Demo
8. Key Takeaways

## 1. Setup & Imports

In [ ]:
import os, sys, warnings
warnings.filterwarnings('ignore')
os.environ['MLFLOW_ALLOW_FILE_STORE'] = 'true'
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 120
print('✅ All imports successful')

## 2. Raw Data Exploration

In [ ]:
DATA_PATH = '../data/bank-additional-full.csv'
if os.path.exists(DATA_PATH):
    df = pd.read_csv(DATA_PATH, sep=';')
else:
    from src.data.generator import generate_synthetic
    X_train, X_test, y_train, y_test = generate_synthetic()
    df = pd.concat([X_train, y_train], axis=1)

print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

## 3. Transformer Deep Dives

In [ ]:
from src.features.transformers import (
    DurationDropper, CategoricalEncoder, EconomicFeatureEngineer,
    CampaignIntensityEncoder, ContactRecencyEncoder
)
from sklearn.model_selection import train_test_split

df_work = df.copy()
if 'y' in df_work.columns:
    y = (df_work['y'] == 'yes').astype(int)
    X = df_work.drop(columns=['y'])
else:
    y = df_work.iloc[:, -1]
    X = df_work.iloc[:, :-1]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train: {X_train.shape} | Test: {X_test.shape}')

In [ ]:
# Transformer 1: DurationDropper
t1 = DurationDropper()
t1.fit(X_train)
X_no_dur = t1.transform(X_train)
print(f'DurationDropper: {X_train.shape[1]} cols -> {X_no_dur.shape[1]} cols')
print(f'Dropped: duration (leakage — unknown before call is made)')

# Show duration correlation with target
if 'duration' in X_train.columns:
    corr = X_train['duration'].corr(y_train)
    print(f'duration correlation with target: {corr:.3f} — HIGH (leakage confirmed)')

In [ ]:
# Transformer 3: EconomicFeatureEngineer
t3 = EconomicFeatureEngineer()
t3.fit(X_no_dur)
X_econ = t3.transform(X_no_dur)
print(f'EconomicFeatureEngineer: {X_no_dur.shape[1]} cols -> {X_econ.shape[1]} cols')
print('New feature: euribor_emp_interaction = euribor3m * emp.var.rate')
print('Business logic: low euribor + negative emp.var.rate = economic downturn = higher subscription')

# Visualise the interaction
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, (col, ax) in enumerate(zip(['euribor3m', 'emp.var.rate', 'euribor_emp_interaction'], axes)):
    if col in X_econ.columns:
        ax.hist(X_econ[col], bins=30, color=['#2563EB','#16A34A','#DC2626'][i], edgecolor='white', alpha=0.8)
        ax.set_title(col, fontweight='bold')
plt.suptitle('Economic Features Distribution', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Transformer 4: CampaignIntensityEncoder
t4 = CampaignIntensityEncoder()
t4.fit(X_econ)
X_camp = t4.transform(X_econ)
print('CampaignIntensityEncoder bins:')
print('  0 = low (1 contact)')
print('  1 = medium (2-3 contacts)')
print('  2 = high (4-6 contacts)')
print('  3 = very_high (7+ contacts = customer fatigue)')
if 'campaign_intensity' in X_camp.columns:
    print(f'\nIntensity distribution:\n{X_camp["campaign_intensity"].value_counts().sort_index()}')

## 4. Full Pipeline Execution

In [ ]:
from src.features.pipeline_builder import fit_and_log_pipeline

mlflow.set_tracking_uri('../mlruns')
pipeline, run_id, X_train_t, X_test_t, y_train_p, y_test_p = fit_and_log_pipeline(
    data_path=DATA_PATH if os.path.exists(DATA_PATH) else None
)
print(f'\nPipeline run ID: {run_id[:8]}...')

## 5. Feature Impact Analysis

In [ ]:
# Compare features before and after
if isinstance(X_train_t, pd.DataFrame):
    new_features = [c for c in X_train_t.columns if c not in X_train.columns]
    removed_features = [c for c in X_train.columns if c not in X_train_t.columns]
    print(f'Input features:   {X_train.shape[1]}')
    print(f'Output features:  {X_train_t.shape[1]}')
    print(f'Added:   {new_features}')
    print(f'Removed: {removed_features}')

    # Feature correlations with target
    if isinstance(y_train_p, pd.Series):
        corrs = X_train_t.corrwith(y_train_p).abs().sort_values(ascending=False)
        fig, ax = plt.subplots(figsize=(10, 6))
        colors_bar = ['#DC2626' if c in new_features else '#2563EB' for c in corrs.index[:15]]
        corrs.head(15).plot(kind='barh', ax=ax, color=colors_bar[::-1], edgecolor='white')
        ax.set_title('Top 15 Feature Correlations with Target\n(Red = new engineered features)', fontweight='bold')
        ax.set_xlabel('Absolute Correlation')
        plt.tight_layout()
        plt.show()

## 6. Model Training on Transformed Features

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, f1_score, classification_report

model = RandomForestClassifier(n_estimators=200, max_depth=10, class_weight='balanced', random_state=42, n_jobs=-1)
model.fit(X_train_t, y_train_p)

y_prob = model.predict_proba(X_test_t)[:, 1]
y_pred = model.predict(X_test_t)

auc = roc_auc_score(y_test_p, y_prob)
f1  = f1_score(y_test_p, y_pred, zero_division=0)

print(f'AUC: {auc:.4f}')
print(f'F1:  {f1:.4f}')
print(f'\nProject 1 baseline AUC: 0.8174')
print(f'Feature pipeline AUC:   {auc:.4f} ({auc-0.8174:+.4f})')
print(f'\n{classification_report(y_test_p, y_pred, target_names=["No","Yes"])}')

## 7. Key Takeaways

In [ ]:
print('=' * 60)
print('KEY TAKEAWAYS — Feature Engineering Pipeline')
print('=' * 60)
print('''
5 TRANSFORMERS
  1. DurationDropper          — removes leakage feature
  2. CategoricalEncoder       — label encodes all objects
  3. EconomicFeatureEngineer  — euribor * emp.var.rate
  4. CampaignIntensityEncoder — bins contact count
  5. ContactRecencyEncoder    — bins pdays into recency

RESULTS (Real Data)
  Input features:  20
  Output features: 22 (2 new engineered)
  AUC: 0.8192 vs baseline 0.8174 (+0.0018)

KEY MLOps CONCEPT — DECOUPLING
  Feature pipeline = separate MLflow run
  Model = separate MLflow run
  Change features without retraining model
  Change model without re-engineering features

NEXT STEPS
  Project 4: Optuna hyperparameter tuning
  Project 6: Feature Store with Feast + Redis
''')
print('=' * 60)

---
**Author:** Jumma Mohammad Teli | Data Analyst & ML Engineer | Birmingham, UK  
**LinkedIn:** https://linkedin.com/in/jumma-mohammad  
**GitHub:** https://github.com/jumma786  

*Project 3 of 10 — MLOps Portfolio Series*